# Train GATv2 — PRDetect

Mirror cua `gcn.py` nhung dung **GATv2Model** (cai tien Phan 4).  
Chay tung cell theo thu tu.

In [ ]:
import os
import pickle

with open('./graph_data/hc3_train.pkl', 'rb') as f:
    hc3_train = pickle.load(f)
with open('./graph_data/hc3_val.pkl', 'rb') as f:
    hc3_val = pickle.load(f)

print(f"Train: {len(hc3_train['y'])} mau")
print(f"Val  : {len(hc3_val['y'])} mau")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from datetime import datetime
from torch_geometric.data import Data
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
import time

from model.GATv2 import GATv2Model

In [ ]:
seed        = 2024
dataset_name = 'hc3'

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- Hyperparameters (giong gcn.py, chi thay model) ---
input_dim  = 768   # RoBERTa embedding
hidden_dim = 64    # Chieu an moi head
output_dim = 64    # Chieu ra
heads      = 4     # So attention heads -> output layer1 = 64*4 = 256
dropout    = 0.5   # Feature dropout (nhu gcn.py)
attn_dropout = 0.1 # Attention dropout (tach biet, cai tien Phan 4)
lr         = 0.0001  # Giong gcn.py
epochs     = 40

print(f'Config: hidden={hidden_dim}, heads={heads}, output={output_dim}')
print(f'        dropout={dropout}, attn_dropout={attn_dropout}, lr={lr}')

In [ ]:
gatv2model = GATv2Model(
    input_dim    = input_dim,
    hidden_dim   = hidden_dim,
    output_dim   = output_dim,
    heads        = heads,
    dropout      = dropout,
    attn_dropout = attn_dropout
).to(device)

optimizer = optim.Adam(gatv2model.parameters(), lr=lr)
criterion = nn.BCELoss()

total_params     = sum(p.numel() for p in gatv2model.parameters())
trainable_params = sum(p.numel() for p in gatv2model.parameters() if p.requires_grad)
print(f'GATv2Model — Total params: {total_params:,}, Trainable: {trainable_params:,}')
print(gatv2model)

In [ ]:
train_len = len(hc3_train['y'])
val_len   = len(hc3_val['y'])

train_loss = []
val_loss   = []
train_acc  = []
val_acc    = []
val_max_acc = -1

log_dir = f'logs/gatv2_{dataset_name}_{seed}_' + datetime.now().strftime('%Y%m%d-%H%M%S')
writer  = SummaryWriter(log_dir)
print(f'TensorBoard log: {log_dir}')

In [ ]:
start_time = time.time()

for epoch in range(epochs):

    # ===== Training set =====
    gatv2model.train()
    epoch_loss = 0.0
    correct_predictions = 0

    for i in tqdm(range(train_len), desc=f'epoch: {epoch+1}, Training'):
        data = Data(
            x          = hc3_train['all_token_embeddings'][i],
            edge_index = hc3_train['all_edge_index'][i],
            y          = hc3_train['y'][i]
        ).to(device)

        optimizer.zero_grad()
        outputs = gatv2model(data)
        loss = criterion(outputs, data.y.float().view(-1, 1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        predictions = (outputs >= 0.5).long()
        correct_predictions += (predictions == data.y.view(-1, 1)).sum().item()

    epoch_loss /= train_len
    writer.add_scalar('Loss/train', epoch_loss, epoch)
    epoch_acc = correct_predictions / train_len
    writer.add_scalar('Acc/train', epoch_acc, epoch)
    print(f'epoch: {epoch+1}, train_loss: {epoch_loss:.4f}, train_acc: {epoch_acc:.4f}')
    train_loss.append(epoch_loss)
    train_acc.append(epoch_acc)

    # ===== Validation set =====
    gatv2model.eval()
    epoch_loss = 0.0
    correct_predictions = 0
    all_predictions = []

    with torch.no_grad():
        for i in tqdm(range(val_len), desc=f'epoch: {epoch+1}, Validation'):
            data = Data(
                x          = hc3_val['all_token_embeddings'][i],
                edge_index = hc3_val['all_edge_index'][i],
                y          = hc3_val['y'][i]
            ).to(device)

            outputs = gatv2model(data)
            loss = criterion(outputs, data.y.float().view(-1, 1))
            epoch_loss += loss.item()
            predictions = (outputs >= 0.5).long()
            all_predictions.append(predictions)
            correct_predictions += (predictions == data.y.view(-1, 1)).sum().item()

    epoch_loss /= val_len
    writer.add_scalar('Loss/val', epoch_loss, epoch)
    epoch_acc = correct_predictions / val_len
    writer.add_scalar('Acc/val', epoch_acc, epoch)
    print(f'epoch: {epoch+1}, val_loss: {epoch_loss:.4f}, val_acc: {epoch_acc:.4f}')
    val_loss.append(epoch_loss)
    val_acc.append(epoch_acc)

    # ===== Save best model =====
    if epoch_acc >= val_max_acc:
        val_max_acc = epoch_acc
        os.makedirs('./model', exist_ok=True)
        torch.save(gatv2model.state_dict(), f'./model/{dataset_name}_gatv2_model_{seed}.pth')
        print(f'  -> Best model saved (val_acc={val_acc[-1]:.4f})')

end_time = time.time()
elapsed_time = end_time - start_time
print(f'\nElapsed time: {elapsed_time:.1f} seconds')
print(f'Best val_acc : {val_max_acc:.4f}')
writer.close()

---
## Testing

In [ ]:
test_file = 'hc3_test'
with open(f'./graph_data/{test_file}.pkl', 'rb') as f:
    hc3_test = pickle.load(f)
test_len = len(hc3_test['y'])
print(f'Test: {test_len} mau')

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score

test_gatv2model = GATv2Model(
    input_dim    = input_dim,
    hidden_dim   = hidden_dim,
    output_dim   = output_dim,
    heads        = heads,
    dropout      = dropout,
    attn_dropout = attn_dropout
).to(device)

test_gatv2model.load_state_dict(
    torch.load(f'./model/{dataset_name}_gatv2_model_{seed}.pth', map_location=device)
)
test_gatv2model.eval()

test_loss = 0.0
correct_predictions = 0
test_pres = []

start_time = time.time()
with torch.no_grad():
    for i in tqdm(range(test_len), desc='Test'):
        data = Data(
            x          = hc3_test['all_token_embeddings'][i],
            edge_index = hc3_test['all_edge_index'][i],
            y          = hc3_test['y'][i]
        ).to(device)

        outputs = test_gatv2model(data)
        test_pres.append(outputs.item())
        loss = criterion(outputs, data.y.float().view(-1, 1))
        test_loss += loss.item()
        predictions = (outputs >= 0.5).long()
        correct_predictions += (predictions == data.y.view(-1, 1)).sum().item()

end_time = time.time()
print(f'Elapsed time: {end_time - start_time:.1f} seconds')

y_pred  = [1 if prob >= 0.5 else 0 for prob in test_pres]
y_true  = hc3_test['y'].view(-1, 1)
test_loss /= test_len
test_acc  = correct_predictions / test_len
test_f1   = f1_score(y_true, y_pred)
auc       = roc_auc_score(hc3_test['y'], test_pres)

print(f'test_loss: {test_loss:.4f}')
print(f'test_acc : {test_acc:.4f}')
print(f'test_f1  : {test_f1:.4f}')
print(f'auc      : {auc:.4f}')

In [ ]:
# Ghi ket qua ra file (giong gcn.py)
with open('test_result.txt', 'a', encoding='utf-8') as w:
    w.write(
        f'{test_file}\t'
        f'acc: {test_acc:.4f}\t'
        f'auc: {auc:.4f}\t'
        f'f1: {test_f1:.4f}\t'
        f'seed: {seed}\t'
        f'model: gatv2_{dataset_name}\t'
        f'{datetime.now()}\n'
    )
print('Da ghi ket qua vao test_result.txt')

---
## (Tuy chon) Phan tich Node Nhieu qua Attention
Dung `get_suspicious_nodes()` de xem node nao bi GATv2 gan attention thap.

In [ ]:
# Lay 1 mau bat ky de xem attention
sample_idx = 0
sample_data = Data(
    x          = hc3_test['all_token_embeddings'][sample_idx],
    edge_index = hc3_test['all_edge_index'][sample_idx],
    y          = hc3_test['y'][sample_idx]
).to(device)

suspicious = test_gatv2model.get_suspicious_nodes(sample_data, threshold=0.05)
prob, (alpha1, alpha2) = test_gatv2model(sample_data, return_attention=True)

print(f'Du doan: {prob.item():.4f} (1=human, 0=AI)')
print(f'Nhan that: {sample_data.y.item()}')
print(f'So node bi nghi la nhieu: {len(suspicious)}')
print(f'Chi so node dang ngo    : {suspicious[:20]}')
print(f'alpha1 shape: {alpha1.shape}  (E x heads)')